# Complementary code for GIFT ICA output analysis

In [ ]:
# imports

import pandas as pd
from statsmodels.stats.multitest import fdrcorrection
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import matplotlib.patches as patches
import matplotlib as mpl

## prepare the data

In [ ]:
# load the spectral power p values for each IC from the excel file

# IC 19
ic19_pval = pd.read_excel(
    "spectral_data.xlsx",
    sheet_name="19",
    usecols="A:DY",
    skiprows=lambda x: x not in [0, 1, 2],  # keep only first 3 rows
    header=None
)

# Rename columns from 1 to 129 (frequency bins)
ic19_pval.columns = range(1, 130)

# Rename rows to "1-2", "1-3", "2-3" (group contrasts)
ic19_pval.index = ["1-2", "1-3", "2-3"]

####################################################################

# IC 29
ic29_pval = pd.read_excel(
    "spectral_data.xlsx",
    sheet_name="29",
    usecols="A:DY",
    skiprows=lambda x: x not in [0, 1, 2],  # keep only first 3 rows
    header=None
)

# Rename columns from 1 to 129 (frequency bins)
ic29_pval.columns = range(1, 130)

# Rename rows to "1-2", "1-3", "2-3"
ic29_pval.index = ["1-2", "1-3", "2-3"]

####################################################################

# IC 46
ic46_pval = pd.read_excel(
    "spectral_data.xlsx",
    sheet_name="46",
    usecols="A:DY",
    skiprows=lambda x: x not in [0, 1, 2],  # keep only first 3 rows
    header=None
)

# Rename columns from 1 to 129 (frequency bins)
ic46_pval.columns = range(1, 130)

# Rename rows to "1-2", "1-3", "2-3" (group contrasts)
ic46_pval.index = ["1-2", "1-3", "2-3"]


In [ ]:
# define functions to seperate by frequency bands
def fdr_then_split_low_high_flat(pval_df, alpha=0.05, cutoff=50):
    """
    For each row (group comparison):
      1) FDR-correct across all bins in that row
      2) keep bins that survive FDR
      3) split survivors into low/high using cutoff
    Returns flat dict with 6 keys: '1-2_low', '1-2_high', etc.
    """
    out = {}

    for comp in pval_df.index:
        pvals = pval_df.loc[comp].astype(float).values
        rejected, _ = fdrcorrection(pvals, alpha=alpha)

        sig_bins = pval_df.columns[rejected].tolist()

        out[f"{comp}_low"]  = [b for b in sig_bins if b <= cutoff]
        out[f"{comp}_high"] = [b for b in sig_bins if b > cutoff]

    return out

def split_low_high_flat_uncorrected(pval_df, alpha=0.05, cutoff=50):
    """
    Same output format as FDR dict using uncorrected p < alpha.
    Returns 6 keys: '1-2_low', '1-2_high', etc.
    """
    out = {}
    for comp in pval_df.index:
        sig_bins = pval_df.columns[pval_df.loc[comp].astype(float) < alpha].tolist()
        out[f"{comp}_low"]  = [b for b in sig_bins if b <= cutoff]
        out[f"{comp}_high"] = [b for b in sig_bins if b > cutoff]
    return out

In [ ]:
# create dicts of significant bins for each IC
fdr_sig_19 = fdr_then_split_low_high_flat(ic19_pval, alpha=0.05, cutoff=50)
fdr_sig_29 = fdr_then_split_low_high_flat(ic29_pval, alpha=0.05, cutoff=50)
fdr_sig_46 = fdr_then_split_low_high_flat(ic46_pval, alpha=0.05, cutoff=50)

# "Suggestive" (uncorrected) bins
unc_sig_19 = split_low_high_flat_uncorrected(ic19_pval, alpha=0.05, cutoff=50)
unc_sig_29 = split_low_high_flat_uncorrected(ic29_pval, alpha=0.05, cutoff=50)
unc_sig_46 = split_low_high_flat_uncorrected(ic46_pval, alpha=0.05, cutoff=50)

In [ ]:
# load the spectral power results for each IC from the excel file

# IC 19
ic19_ps = pd.read_excel(
    "spectral_data.xlsx",
    sheet_name="19",
    usecols="A:DY",
    skiprows=range(0, 4),  # skip first 4 rows (rows 0–3)
    nrows=38,              # then read next 38 rows (rows 5–42)
    header=None

)

# Rename columns from 1 to 129
ic19_ps.columns = range(1, 130)

# Rename rows to from 1 to 38
ic19_ps.index = range(1, 39)

####################################################################

# IC 29
ic29_ps = pd.read_excel(
    "spectral_data.xlsx",
    sheet_name="29",
    usecols="A:DY",
    skiprows=range(0, 4),  # skip first 4 rows (rows 0–3)
    nrows=38,              # then read next 38 rows (rows 5–42)
    header=None

)

# Rename columns from 1 to 129
ic29_ps.columns = range(1, 130)

ic29_ps.index = range(1, 39)

####################################################################

# IC 46
ic46_ps = pd.read_excel(
    "spectral_data.xlsx",
    sheet_name="46",
    usecols="A:DY",
    skiprows=range(0, 4),  # skip first 4 rows (rows 0–3)
    nrows=38,              # then read next 38 rows (rows 5–42)
    header=None

)

# Rename columns from 1 to 129
ic46_ps.columns = range(1, 130)

ic46_ps.index = range(1, 39)

In [ ]:
# load covariates
covar_df = pd.read_excel(
    "spectral_data.xlsx",
    sheet_name="confounds",
    usecols="A:H",
    nrows=39                # then read next 38 rows (rows 5–42)
    #header=None

)

covar_df.groupby('group')['mli'].agg(['mean', 'std', 'count']) # group by group and get mean, std, and count for medication load index

In [ ]:
# load the frequency bins from the excel file

# frequency
hz = pd.read_excel(
    "spectral_data.xlsx",
    sheet_name="29",
    usecols="A:DY",
    skiprows=range(0, 43),  # skip first 4 rows (rows 0–3)
    nrows=1,                # then read next 38 rows (rows 5–42)
    header=None

)

hz=hz.T

# Rename columns from 1 to 129
hz.columns = ["hz"]

hz.index = range(1, 130)

## Sensetivity checks - leave one out aasocation betwen ps and ref-rem group diff

In [ ]:
# Filter covariates to only those needed for the model
cov_use = covar_df[["group", "age", "sex", "madrs"]].copy()  
# make group a clean string/categorical
cov_use["group"] = cov_use["group"].astype(str).str.strip()

# compute per-subject mean power for each (comp_band)
# using FDR-significant bin indices dict (e.g., fdr_sig_29)
def band_means_from_bins(power_df, bins_dict):
    """
    power_df: participants x bins (cols 1..129)
    bins_dict: keys like '2-3_low', values = list of bin indices
    returns: DataFrame participants x keys with mean power across those bins
    """
    out = {}
    for k, bins in bins_dict.items():
        if len(bins) == 0:
            out[k] = pd.Series(np.nan, index=power_df.index)
        else:
            out[k] = power_df[bins].mean(axis=1)
    return pd.DataFrame(out)

# Fit the two models for a given comparison (low + high)
def fit_low_high_models(df, comp, covars=("age", "sex", "madrs")):
    g1, g2 = comp_map[comp]
    sub = df[df["group"].isin([g1, g2])].copy()

    models = {}
    for band in ["low", "high"]:
        y = f"{comp}_{band}"
        if sub[y].isna().all():
            models[y] = None
            continue
        formula = "Q('{}') ~ C(group)".format(y)
        if covars:
            formula += " + " + " + ".join(covars)
        models[y] = smf.ols(formula, data=sub).fit()
    return models


# Leave-one-out for those two outcomes (comp_low + comp_high)
# Leave out participants from ONE group 
def loo_low_high(df, comp, leave_group, covars=("age", "sex", "madrs")):
    g1, g2 = comp_map[comp]
    sub = df[df["group"].isin([g1, g2])].copy()

    leave_idx = sub.index[sub["group"] == leave_group]
    rows = []

    for band in ["low", "high"]:
        y = f"{comp}_{band}"
        if sub[y].isna().all():
            continue

        formula = "Q('{}') ~ C(group)".format(y)
        if covars:
            formula += " + " + " + ".join(covars)

        for idx in leave_idx:
            sub_loo = sub.drop(index=idx)
            m = smf.ols(formula, data=sub_loo).fit()
            group_terms = {k: v for k, v in m.params.items() if k.startswith("C(group)[T.")}
            group_pvals = {k: v for k, v in m.pvalues.items() if k.startswith("C(group)[T.")}

            rows.append({
                "comp": comp,
                "band": band,
                "left_out": idx,
                "group_terms_beta": group_terms,
                "group_terms_p": group_pvals
            })

    return pd.DataFrame(rows)

In [ ]:
# perform LOO analysis for IC 29

means_29 = band_means_from_bins(ic29_ps, fdr_sig_29)
# Merge with covariates
df29 = pd.concat([cov_use, means_29], axis=1)

# Map comparison codes to the two groups involved
print("Group labels found:", cov_use["group"].unique())

comp_map = {
    "1-2": ("1", "2"),
    "1-3": ("1", "3"),
    "2-3": ("2", "3"),
}

models_29_23 = fit_low_high_models(df29, "2-3", covars=("age", "sex", "madrs"))
# models_29_23["2-3_low"].summary()
# models_29_23["2-3_high"].summary()

# Example: for comp 2-3, leave out group "2"
loo_29_23 = loo_low_high(df29, "2-3", leave_group="2", covars=("age", "sex", "madrs"))


In [ ]:
# view results

# copy to avoid modifying in place
loo = loo_29_23.copy()

# extract beta (first and only value in the dict)
loo["beta"] = loo["group_terms_beta"].apply(
    lambda d: np.nan if not isinstance(d, dict) or len(d) == 0 else list(d.values())[0]
)

# extract p-value (matching key)
loo["p"] = loo["group_terms_p"].apply(
    lambda d: np.nan if not isinstance(d, dict) or len(d) == 0 else list(d.values())[0]
)

# optional: also keep the term name
loo["group_term"] = loo["group_terms_beta"].apply(
    lambda d: None if not isinstance(d, dict) or len(d) == 0 else list(d.keys())[0]
)


In [ ]:
# perform LOO analysis for IC 19

means_19 = band_means_from_bins(ic19_ps, fdr_sig_19)
# Merge with covariates
df19 = pd.concat([cov_use, means_19], axis=1)
models_19_23 = fit_low_high_models(df19, "2-3", covars=("age", "sex", "madrs"))
loo_19_23 = loo_low_high(df19, "2-3", leave_group="3", covars=("age", "sex", "madrs"))

In [ ]:
# view results
loo = loo_19_23.copy()

# extract beta (first and only value in the dict)
loo["beta"] = loo["group_terms_beta"].apply(
    lambda d: np.nan if not isinstance(d, dict) or len(d) == 0 else list(d.values())[0]
)

# extract p-value (matching key)
loo["p"] = loo["group_terms_p"].apply(
    lambda d: np.nan if not isinstance(d, dict) or len(d) == 0 else list(d.values())[0]
)

# optional: also keep the term name
loo["group_term"] = loo["group_terms_beta"].apply(
    lambda d: None if not isinstance(d, dict) or len(d) == 0 else list(d.keys())[0]
)


## sensetivity- correlations within ref and across all with madrs, hama, age

In [ ]:
# define a function to compute correlations between covariates and IC power for specified bins, with FDR correction
def corr_bins_fdr(ic_ps, bins_lookup, cov_df,# group_series,
                  comp, band, cov_name,
                  group_num=2, alpha=0.05, only_sig=False):

    key = f"{comp}_{band.lower()}"
    bins = bins_lookup.get(key, [])
    if len(bins) == 0:
        return pd.DataFrame(), None


    #### TO RUN ON SPECIFIC GROUP
  # group subset + align indices
  #  idx = group_series[group_series.astype(str) == str(group_num)].index
  #  idx = idx.intersection(ic_ps.index).intersection(cov_df.index)

    #### TO RUN ON ALL SUBJECTS
    idx = ic_ps.index.intersection(cov_df.index)

    if len(idx) == 0:
        return pd.DataFrame(), None

    # force matching bin labels
    ic_ps = ic_ps.copy()
    ic_ps.columns = ic_ps.columns.astype(str)
    bins = [str(b) for b in bins if str(b) in ic_ps.columns]
    if len(bins) == 0:
        return pd.DataFrame(), None

    x = cov_df.loc[idx, cov_name].astype(float)

    rows = []
    for b in bins:
        y = ic_ps.loc[idx, b].astype(float)
        tmp = pd.concat([x, y], axis=1).dropna()
        if tmp.shape[0] < 3:
            continue
        r, p = pearsonr(tmp.iloc[:, 0].values, tmp.iloc[:, 1].values)
        rows.append((b, tmp.shape[0], r, p))

    if len(rows) == 0:
        return pd.DataFrame(), None

    res = pd.DataFrame(rows, columns=["bin", "n", "r", "p"])
    sig, p_fdr = fdrcorrection(res["p"].values, alpha=alpha)
    res["p_fdr"] = p_fdr
    res["sig"] = sig

    use = res[res["sig"]] if only_sig else res
    summ = None if use.empty else {
        "comp_band": key,
        "covariate": cov_name,
        "group_num": int(group_num),
        "n_bins": int(use.shape[0]),
        "avg_r": float(use["r"].mean()),
        "avg_r2": float((use["r"]**2).mean()),
        "avg_p": float(use["p"].mean()),
        "avg_p_fdr": float(use["p_fdr"].mean()),
    }

    return res, summ


In [ ]:
# dictionaries
ic_ps_dict = {"19": ic19_ps, "29": ic29_ps, "46": ic46_ps}
sigbins_dict = {"19": fdr_sig_19, "29": fdr_sig_29, "46": fdr_sig_46}

# inputs
group_series = covar_df["group"]
cov_df_use = covar_df[["madrs", "hama", "age", "mli", "gad", "dx"]]

ic_key = "46"
ic_ps = ic_ps_dict[ic_key]
bins_lookup = sigbins_dict[ic_key]

res, summ = corr_bins_fdr(
    ic_ps, bins_lookup, cov_df_use, group_series,
    comp="1-2", band="high", cov_name="dx",
    group_num=2, alpha=0.05, only_sig=False
)

#### currently runing on all participants, change to group specific in function itself

## plots

### Figure 1 multivariate

In [ ]:
file1 = 'sm_multi.csv'
file2 = 'ps_multi.csv'

df_raw1 = pd.read_csv(file1, index_col=['Network', 'Component'])
df_raw2 = pd.read_csv(file2, index_col=['Network', 'Component'])

In [ ]:
width_npp = 7.08661

fig, ax = plt.subplots(
    figsize=(width_npp, 2.5),
    ncols=2,
    gridspec_kw={'width_ratios': [0.8, 1], 'wspace': -0.77}
)

mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.size'] = 5  


# Load and preprocess the first dataframe
df1 = pd.read_csv(file1, index_col=['Network', 'Component'])
#df1=df1.iloc[:,:-1]
df1 = df1.T
df1 = -np.log10(df1)

# Load and preprocess the second dataframe
df2 = pd.read_csv(file2, index_col=['Network', 'Component'])
#df2=df2.iloc[:,:-1]
df1 = df1.T
df2 = -np.log10(df2)

# Sort and index the dataframes by network and component
df1 = df1.sort_index(level=['Network', 'Component'])
df2 = df2.sort_index(level=['Network', 'Component'])

# Combine all dataframes to find the overall minimum and maximum values (so colorbar can be used for both plots)
combined_df = pd.concat([df1.stack(), df2.stack()])
overall_min = combined_df.min()
overall_max = combined_df.max()

# Create hierarchical y-axis labels so outer label is network and inner label is component number
networks = df1.index.get_level_values('Network')
components = df1.index.get_level_values('Component')
y_labels_inner = [f"{comp}" for comp in components]

In [ ]:
# Plot the first heatmap (SM)
sns.heatmap(df1, annot=False, cmap="YlOrRd",
            mask=df1.isna(), linewidths=.5, linecolor='grey', ax=ax[0], square=True, cbar=False, vmin=overall_min, vmax=overall_max)
#ax[0].set_yticks([])
ax[0].set_yticks(np.arange(df1.shape[0]) + 0.5)
ax[0].set_yticklabels(y_labels_inner, ha="right", fontsize=5)
ax[0].tick_params(axis='y', pad=0)  # increase pad value to move labels farther from axis
ax[0].set_xticks(np.arange(df1.shape[1]) + 0.5)
ax[0].set_xticklabels(df1.columns, ha="right", rotation=45, fontsize=5)
ax[0].set_title("Spatial Maps", fontsize=5, loc='left', weight='bold')
ax[0].set_xlabel("")  # Remove x-axis label
ax[0].set_ylabel("")  # Remove y-axis label

# Add borders and network labels around tick labels on the left plot
prev_network = None
network_start = 0
for i, (network, component) in enumerate(df1.index):
    if network != prev_network and prev_network is not None:
        # Skip drawing rectangles for the outer labels
        if i > 0:
            # Draw a rectangle around the group of components
            rect = patches.Rectangle((-1.7, network_start - 0), 1.7, i - network_start + 0.0, linewidth=0.5, 
                                     edgecolor='black', facecolor='none', transform=ax[0].transData, clip_on=False)
            ax[0].add_patch(rect)
        # Add network label
        mid_pos = (network_start + i - 1) / 2
        ax[0].text(-2.6, mid_pos, prev_network, ha='right', va='center_baseline', 
                   fontsize=5, rotation=45)#, bbox=dict(facecolor='white', edgecolor='none'))
        network_start = i
    prev_network = network
# Draw a rectangle around the last group
rect = patches.Rectangle((-1.7, network_start - 0.), 1.7, len(df1.index) - network_start + 0.0, linewidth=0.5, 
                         edgecolor='black', facecolor='none', transform=ax[0].transData, clip_on=False)
ax[0].add_patch(rect)
# Add the last network label
mid_pos = (network_start + len(df1.index) - 1) / 2
ax[0].text(-2.6, mid_pos, prev_network, ha='right', va='center_baseline', fontsize=5, 
           rotation=45)

# Add border lines to the first plot
for spine in ax[0].spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

# Plot the second heatmap (PS)
sns.heatmap(df2, annot=False, cmap="YlOrRd",
            mask=df2.isna(), linewidths=0.5, linecolor='grey', ax=ax[1], square=True, cbar=False, vmin=overall_min, vmax=overall_max)
ax[1].set_yticks([])  # Remove y-ticks
ax[1].set_yticklabels([])  # Remove y-tick labels
ax[1].set_xticks(np.arange(df2.shape[1]) + 0.5)
ax[1].set_xticklabels(df2.columns, ha="right", rotation=45, fontsize=5)
ax[1].set_title("Power Spectra", fontsize=5, loc='left', weight='bold')
ax[1].set_xlabel("")  # Remove x-axis label
ax[1].set_ylabel("")  # Remove y-axis label

# Add border lines to the second plot
for spine in ax[1].spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

# Adjust layout of the plots
plt.subplots_adjust(left=0.4, right=0.9, top=0.9, bottom=0.1)

# Add a shared x-axis label for the two plots
fig.text(0.61, -0.15, "Covariates", ha='center', fontsize=5)

# Add a color bar
cbar_ax = fig.add_axes([0.73, 0.15, 0.008, 0.63])  # Position color bar
cbar = plt.colorbar(ax[0].collections[0], cax=cbar_ax)
cbar.set_label(r'$-\log_{10}(\mathit{p})$', fontsize=5)
cbar.ax.tick_params(labelsize=5) 


# Save the combined plot
plt.savefig(
    'multi_sweet_sans_serif.pdf',
    dpi=400,
    bbox_inches='tight',
    facecolor='white'
)
plt.show()

### Figure 2 univariate

#### IC29

In [ ]:
# Plotting figure 2

# X axis: frequencies
freqs = hz["hz"].to_numpy()

# Group labels per participant 
# align indices
covar_plot = covar_df.copy()
covar_plot.index = covar_plot.index + 1  # 0..37 -> 1..38

idx = ic29_ps.index.intersection(covar_plot.index)

ic29_ps_aligned = ic29_ps.loc[idx]
group_labels = covar_plot.loc[idx, "group"].astype(int)

# Map group numbers to legend labels
group_map = {1: "HC", 2: "REM", 3: "REF"}

colors = {
    1: "#20a39e",  
    2: "#ef5b5b",
    3: "#ffba49"
}

plt.figure(figsize=(10, 6))


# Plot mean ± SEM for each group 
for g in sorted(group_labels.unique()):
    rows = group_labels[group_labels == g].index
    ps_group = ic29_ps_aligned.loc[rows]

    mean_ps = ps_group.mean(axis=0).to_numpy()
    sem_ps  = ps_group.sem(axis=0).to_numpy()

    plt.plot(freqs, mean_ps, label=group_map.get(g, str(g)), color=colors.get(g, "k"), linewidth=2)
    plt.fill_between(freqs, mean_ps - sem_ps, mean_ps + sem_ps, color=colors.get(g, "k"), alpha=0.2)


# Legend entries for significance annotations
custom_lines = [
    Line2D([0], [0], color='grey', marker='*', linestyle='None', markersize=6,
           label='HC vs REM, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='dimgrey', marker='*', linestyle='None', markersize=6,
           label='HC vs REF, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='k', marker='*', linestyle='None', markersize=6,
           label='REF vs REM, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='grey', linestyle='-', linewidth=2,
           label='HC vs REM, $p_{suggestive}<0.05$'),
    Line2D([0], [0], color='dimgrey', linestyle='-', linewidth=2,
           label='HC vs REF, $p_{suggestive}<0.05$'),
    Line2D([0], [0], color='k', linestyle='-', linewidth=2,
           label='REF vs REM, $p_{suggestive}<0.05$')
]

#handles, labels = plt.gca().get_legend_handles_labels()
#plt.legend(handles=handles + custom_lines, fontsize=10)

# Axis labels 
plt.xlabel("Frequency (Hz)", fontsize=12)
plt.ylabel("Spectral Power", fontsize=12)
plt.tight_layout()
plt.grid(False)

for spine in plt.gca().spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.5)


# draw markers for significant bins
def draw_bin_markers(sig_bins, y, text, color, fontsize):
    for b in sig_bins:
        if 1 <= b <= len(freqs):
            plt.text(freqs[b - 1], y, text, color=color, ha='center', va='bottom', fontsize=fontsize)

# Low frequency markers
draw_bin_markers(unc_sig_29.get("1-3_low", []),  -0.0006, "-", "#7f7f7f", 15)
draw_bin_markers(unc_sig_29.get("1-2_low", []),  -0.0012, "-", "#595959", 15)
draw_bin_markers(unc_sig_29.get("2-3_low", []),  -0.0018, "-", "k",       15)

draw_bin_markers(fdr_sig_29.get("1-3_low", []),  -0.0006, "*", "#7f7f7f", 10)
draw_bin_markers(fdr_sig_29.get("1-2_low", []),  -0.0012, "*", "#595959", 10)
draw_bin_markers(fdr_sig_29.get("2-3_low", []),  -0.0018, "*", "k",       10)

# High frequency markers
draw_bin_markers(unc_sig_29.get("1-3_high", []), -0.0003, "-", "#7f7f7f", 15)
draw_bin_markers(unc_sig_29.get("1-2_high", []), -0.0009, "-", "#595959", 15)
draw_bin_markers(unc_sig_29.get("2-3_high", []), -0.0015, "-", "k",       15)

draw_bin_markers(fdr_sig_29.get("1-3_high", []), -0.0003, "*", "#7f7f7f", 10)
draw_bin_markers(fdr_sig_29.get("1-2_high", []), -0.0009, "*", "#595959", 10)
draw_bin_markers(fdr_sig_29.get("2-3_high", []), -0.0015, "*", "k",       10)

plt.savefig("IC29_power_spectrum.svg", format="svg", bbox_inches="tight", dpi=300)
plt.show()


#### IC19

In [ ]:
freqs = hz["hz"].to_numpy()

covar_plot = covar_df.copy()
covar_plot.index = covar_plot.index + 1  # 0..37 -> 1..38

idx = ic19_ps.index.intersection(covar_plot.index)

ic19_ps_aligned = ic19_ps.loc[idx]
group_labels = covar_plot.loc[idx, "group"].astype(int)

group_map = {1: "HC", 2: "REF", 3: "REM"}
colors = {
    1: "#20a39e",  
    2: "#ef5b5b",  
    3: "#ffba49"  
}

plt.figure(figsize=(10, 6))

for g in sorted(group_labels.unique()):
    rows = group_labels[group_labels == g].index
    ps_group = ic19_ps_aligned.loc[rows]

    mean_ps = ps_group.mean(axis=0).to_numpy()
    sem_ps  = ps_group.sem(axis=0).to_numpy()

    plt.plot(freqs, mean_ps, label=group_map.get(g, str(g)), color=colors.get(g, "k"), linewidth=2)
    plt.fill_between(freqs, mean_ps - sem_ps, mean_ps + sem_ps, color=colors.get(g, "k"), alpha=0.2)

custom_lines = [
    Line2D([0], [0], color='grey', marker='*', linestyle='None', markersize=6,
           label='HC vs REM, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='dimgrey', marker='*', linestyle='None', markersize=6,
           label='HC vs REF, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='k', marker='*', linestyle='None', markersize=6,
           label='REF vs REM, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='grey', linestyle='-', linewidth=2,
           label='HC vs REM, $p_{suggestive}<0.05$'),
    Line2D([0], [0], color='dimgrey', linestyle='-', linewidth=2,
           label='HC vs REF, $p_{suggestive}<0.05$'),
    Line2D([0], [0], color='k', linestyle='-', linewidth=2,
           label='REF vs REM, $p_{suggestive}<0.05$')
]

plt.xlabel("Frequency (Hz)", fontsize=12)
plt.ylabel("Spectral Power", fontsize=12)
plt.tight_layout()
plt.grid(False)

for spine in plt.gca().spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.5)

def draw_bin_markers(sig_bins, y, text, color, fontsize):
    for b in sig_bins:
        if 1 <= b <= len(freqs):
            plt.text(freqs[b - 1], y, text, color=color, ha='center', va='bottom', fontsize=fontsize)


draw_bin_markers(unc_sig_19.get("1-3_low", []),  -0.0006, "-", "#7f7f7f", 15)
draw_bin_markers(unc_sig_19.get("1-2_low", []),  -0.0012, "-", "#595959", 15)
draw_bin_markers(unc_sig_19.get("2-3_low", []),  -0.0018, "-", "k",       15)

draw_bin_markers(fdr_sig_19.get("1-3_low", []),  -0.0006, "*", "#7f7f7f", 10)
draw_bin_markers(fdr_sig_19.get("1-2_low", []),  -0.0012, "*", "#595959", 10)
draw_bin_markers(fdr_sig_19.get("2-3_low", []),  -0.0018, "*", "k",       10)

draw_bin_markers(unc_sig_19.get("1-3_high", []), -0.0003, "-", "#7f7f7f", 15)
draw_bin_markers(unc_sig_19.get("1-2_high", []), -0.0009, "-", "#595959", 15)
draw_bin_markers(unc_sig_19.get("2-3_high", []), -0.0015, "-", "k",       15)

draw_bin_markers(fdr_sig_19.get("1-3_high", []), -0.0003, "*", "#7f7f7f", 10)
draw_bin_markers(fdr_sig_19.get("1-2_high", []), -0.0009, "*", "#595959", 10)
draw_bin_markers(fdr_sig_19.get("2-3_high", []), -0.0015, "*", "k",       10)

plt.savefig("IC19_power_spectrum.svg", format="svg", bbox_inches="tight", dpi=300)

plt.show()


#### IC46

In [ ]:
freqs = hz["hz"].to_numpy()

covar_plot = covar_df.copy()
covar_plot.index = covar_plot.index + 1  # 0..37 -> 1..38

idx = ic46_ps.index.intersection(covar_plot.index)

ic46_ps_aligned = ic46_ps.loc[idx]
group_labels = covar_plot.loc[idx, "group"].astype(int)

group_map = {1: "HC", 2: "REF", 3: "REM"}

colors = {
    1: "#20a39e",  
    2: "#ef5b5b",  
    3: "#ffba49"  
}

plt.figure(figsize=(10, 6))

for g in sorted(group_labels.unique()):
    rows = group_labels[group_labels == g].index
    ps_group = ic46_ps_aligned.loc[rows]

    mean_ps = ps_group.mean(axis=0).to_numpy()
    sem_ps  = ps_group.sem(axis=0).to_numpy()

    plt.plot(freqs, mean_ps, label=group_map.get(g, str(g)), color=colors.get(g, "k"), linewidth=2)
    plt.fill_between(freqs, mean_ps - sem_ps, mean_ps + sem_ps, color=colors.get(g, "k"), alpha=0.2)

custom_lines = [
    Line2D([0], [0], color='grey', marker='*', linestyle='None', markersize=6,
           label='HC vs REM, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='dimgrey', marker='*', linestyle='None', markersize=6,
           label='HC vs REF, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='k', marker='*', linestyle='None', markersize=6,
           label='REF vs REM, $p_{FDR}<0.05$'),
    Line2D([0], [0], color='grey', linestyle='-', linewidth=2,
           label='HC vs REM, $p_{suggestive}<0.05$'),
    Line2D([0], [0], color='dimgrey', linestyle='-', linewidth=2,
           label='HC vs REF, $p_{suggestive}<0.05$'),
    Line2D([0], [0], color='k', linestyle='-', linewidth=2,
           label='REF vs REM, $p_{suggestive}<0.05$')
]

plt.xlabel("Frequency (Hz)", fontsize=12)
plt.ylabel("Spectral Power", fontsize=12)
plt.tight_layout()
plt.grid(False)

for spine in plt.gca().spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.5)

def draw_bin_markers(sig_bins, y, text, color, fontsize):
    for b in sig_bins:
        if 1 <= b <= len(freqs):
            plt.text(freqs[b - 1], y, text, color=color, ha='center', va='bottom', fontsize=fontsize)

draw_bin_markers(unc_sig_46.get("1-3_low", []),  -0.0006, "-", "#7f7f7f", 15)
draw_bin_markers(unc_sig_46.get("1-2_low", []),  -0.0012, "-", "#595959", 15)
draw_bin_markers(unc_sig_46.get("2-3_low", []),  -0.0018, "-", "k",       15)

draw_bin_markers(fdr_sig_46.get("1-3_low", []),  -0.0006, "*", "#7f7f7f", 10)
draw_bin_markers(fdr_sig_46.get("1-2_low", []),  -0.0012, "*", "#595959", 10)
draw_bin_markers(fdr_sig_46.get("2-3_low", []),  -0.0018, "*", "k",       10)

draw_bin_markers(unc_sig_46.get("1-3_high", []), -0.0003, "-", "#7f7f7f", 15)
draw_bin_markers(unc_sig_46.get("1-2_high", []), -0.0009, "-", "#595959", 15)
draw_bin_markers(unc_sig_46.get("2-3_high", []), -0.0015, "-", "k",       15)

draw_bin_markers(fdr_sig_46.get("1-3_high", []), -0.0003, "*", "#7f7f7f", 10)
draw_bin_markers(fdr_sig_46.get("1-2_high", []), -0.0009, "*", "#595959", 10)
draw_bin_markers(fdr_sig_46.get("2-3_high", []), -0.0015, "*", "k",       10)

plt.savefig("IC46_power_spectrum.svg", format="svg", bbox_inches="tight", dpi=300)

plt.show()
